# AlphaMissense Fetch

![AlphaMissense Fetch](https://proto-bio.github.io/proto-assets/images/tool/alphamissense_db/hero.png)

This notebook demonstrates `run_alphamissense_db_fetch`, which retrieves per-residue, per-substitution AlphaMissense pathogenicity scores for human proteins from the AlphaFold Protein Structure Database (AlphaFold DB) by UniProt accession. AlphaMissense is a deep learning model that predicts the pathogenicity of all possible single amino acid substitutions in human proteins, distinguishing likely benign from likely pathogenic missense variants. See the [Cheng et al. 2023 paper](https://doi.org/10.1126/science.adg7492) and the [AlphaMissense GitHub](https://github.com/google-deepmind/alphamissense) for details on the model and its training.

In [1]:
from proto_tools.tools.database_retrieval import (
    AlphaMissenseDBFetchConfig,
    AlphaMissenseDBFetchInput,
    run_alphamissense_db_fetch,
)

## Input API Reference

| Field | Type | Required | Description |
|-------|------|----------|-------------|
| `uniprot_id` | `str` | yes | UniProt accession (human; e.g. 'P04637') |

## Config API Reference

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `coordinate_system` | `Literal["uniprot", "hg19", "hg38"]` | `"uniprot"` | Which AFDB CSV to fetch. UniProt mode returns the full saturation grid in protein coordinates (~7,500 rows for TP53); hg19/hg38 modes return only SNV-accessible substitutions in genomic coordinates and additionally populate `chrom`, `pos`, `ref`, `alt`, `transcript_id` on each prediction. |

The wrapper exposes no filter settings because the AFDB CSV is a static file with no server-side filtering. Filter the returned `predictions` list client-side instead — see the examples below.

## Output API Reference

| Field | Type | Description |
|-------|------|-------------|
| `uniprot_accession` | `str` | UniProt accession looked up |
| `predictions` | `list[AlphaMissensePrediction]` | Per-substitution pathogenicity predictions |
| `num_predictions` | `int` | Number of predictions in the source CSV |
| `mean_pathogenicity` | `float \| None` | Mean pathogenicity score across all predictions |
| `source_url` | `str` | URL of the AlphaMissense CSV fetched |

### `AlphaMissensePrediction` fields

| Field | Type | Mode | Description |
|-------|------|------|-------------|
| `position` | `int` | both | 1-indexed residue position in the canonical UniProt sequence |
| `wild_type_aa` | `str` | both | Single-letter wild-type amino acid at this position |
| `alt_aa` | `str` | both | Single-letter alternate amino acid being scored |
| `pathogenicity_score` | `float` | both | AlphaMissense pathogenicity score (0.0-1.0) |
| `classification` | `AlphaMissenseClass` | both | Class label: 'likely_benign', 'ambiguous', or 'likely_pathogenic' |
| `chrom` | `str \| None` | hg19/hg38 | Chromosome (e.g. `"chr17"`); `None` in UniProt mode |
| `pos` | `int \| None` | hg19/hg38 | 1-indexed genomic position; `None` in UniProt mode |
| `ref` | `str \| None` | hg19/hg38 | Reference allele; `None` in UniProt mode |
| `alt` | `str \| None` | hg19/hg38 | Alternate allele; `None` in UniProt mode |
| `transcript_id` | `str \| None` | hg19/hg38 | GENCODE transcript ID; `None` in UniProt mode |

## Basic Usage

Fetch all AlphaMissense predictions for human TP53 (UniProt accession `P04637`). The full saturation-mutagenesis grid covers every residue in the canonical sequence times the 19 non-self alternate amino acids.

In [2]:
inputs = AlphaMissenseDBFetchInput(uniprot_id="P04637")
config = AlphaMissenseDBFetchConfig()

output = run_alphamissense_db_fetch(inputs, config)

print(f"UniProt accession: {output.uniprot_accession}")
print(f"Predictions in CSV: {output.num_predictions}")
if output.mean_pathogenicity is not None:
    print(f"Mean pathogenicity: {round(output.mean_pathogenicity, 4)}")

print("\nFirst 5 predictions:")
for p in output.predictions[:5]:
    print(f"  {p.wild_type_aa}{p.position}{p.alt_aa}: {round(p.pathogenicity_score, 4)} ({p.classification})")

UniProt accession: P04637
Predictions in CSV: 7467
Mean pathogenicity: 0.5267

First 5 predictions:
  M1A: 0.4065 (ambiguous)
  M1C: 0.511 (ambiguous)
  M1D: 0.8213 (likely_pathogenic)
  M1E: 0.5521 (ambiguous)
  M1F: 0.3445 (ambiguous)


## Pathogenic Variants Only

Filter the saturation grid to high-confidence pathogenic variants client-side after the fetch. The wrapper itself returns every row in the CSV; you slice it.

In [3]:
pathogenic = [p for p in output.predictions if p.pathogenicity_score >= 0.7]

print(f"Total predictions: {output.num_predictions}")
print(f"Pathogenic hits (score >= 0.7): {len(pathogenic)}")

print("\nFirst 5 high-pathogenicity hits:")
for p in pathogenic[:5]:
    print(f"  {p.wild_type_aa}{p.position}{p.alt_aa}: {round(p.pathogenicity_score, 4)} ({p.classification})")

Total predictions: 7467
Pathogenic hits (score >= 0.7): 3014

First 5 high-pathogenicity hits:
  M1D: 0.8213 (likely_pathogenic)
  M1H: 0.728 (likely_pathogenic)
  M1P: 0.7101 (likely_pathogenic)
  E11W: 0.7271 (likely_pathogenic)
  L14D: 0.927 (likely_pathogenic)


## Specific Positions

Restrict the result to specific 1-indexed residue positions. Below we query the three classic TP53 hotspot residues: R175, R248, and R273. Each position returns 19 alternate amino acid substitutions.

In [4]:
hotspots = {175, 248, 273}
hotspot_predictions = [p for p in output.predictions if p.position in hotspots]

print(f"Returned predictions: {len(hotspot_predictions)}")

by_position: dict[int, list] = {}
for p in hotspot_predictions:
    by_position.setdefault(p.position, []).append(p)

for position in sorted(by_position):
    preds = by_position[position]
    wild_type = preds[0].wild_type_aa
    print(f"\nPosition {position} (wild-type {wild_type}):")
    for p in preds:
        print(f"  {p.wild_type_aa}{p.position}{p.alt_aa}: {round(p.pathogenicity_score, 4)} ({p.classification})")

Returned predictions: 57

Position 175 (wild-type R):
  R175A: 0.9957 (likely_pathogenic)
  R175C: 0.9186 (likely_pathogenic)
  R175D: 0.9996 (likely_pathogenic)
  R175E: 0.9975 (likely_pathogenic)
  R175F: 0.9995 (likely_pathogenic)
  R175G: 0.9953 (likely_pathogenic)
  R175H: 0.9857 (likely_pathogenic)
  R175I: 0.9939 (likely_pathogenic)
  R175K: 0.9728 (likely_pathogenic)
  R175L: 0.9907 (likely_pathogenic)
  R175M: 0.999 (likely_pathogenic)
  R175N: 0.9985 (likely_pathogenic)
  R175P: 0.9906 (likely_pathogenic)
  R175Q: 0.9764 (likely_pathogenic)
  R175S: 0.9988 (likely_pathogenic)
  R175T: 0.997 (likely_pathogenic)
  R175V: 0.9898 (likely_pathogenic)
  R175W: 0.9938 (likely_pathogenic)
  R175Y: 0.9977 (likely_pathogenic)

Position 248 (wild-type R):
  R248A: 0.9999 (likely_pathogenic)
  R248C: 0.9978 (likely_pathogenic)
  R248D: 0.9996 (likely_pathogenic)
  R248E: 0.9988 (likely_pathogenic)
  R248F: 0.9999 (likely_pathogenic)
  R248G: 0.9985 (likely_pathogenic)
  R248H: 0.996 (lik

## Class-Based Filter

Restrict the output to a specific AlphaMissense class label by filtering `output.predictions` client-side. Below we keep only `likely_benign` substitutions.

In [5]:
benign = [p for p in output.predictions if p.classification == "likely_benign"]

print(f"Likely-benign predictions: {len(benign)}")

print("\nSample predictions:")
for p in benign[:3]:
    print(f"  {p.wild_type_aa}{p.position}{p.alt_aa}: {round(p.pathogenicity_score, 4)} ({p.classification})")

Likely-benign predictions: 3287

Sample predictions:
  M1L: 0.1774 (likely_benign)
  M1V: 0.1116 (likely_benign)
  E2A: 0.1089 (likely_benign)


## Export Results

Fetched results can be saved to JSON format for downstream use in other tools or analysis pipelines.

In [6]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

# Export the full saturation grid for downstream use.
output.export("alphamissense_db_predictions", export_path=output_dir, file_format="json")
print(f"Exported to {output_dir / 'alphamissense_db_predictions.json'}")

Exported to example_output/alphamissense_db_predictions.json
